# ABCfold backend capability exploration -- one cell per model, pooled across all proteins

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

`tm_conformation_clustering.ipynb` is organised **per protein** (does backend
X agree with backend Y on *this* pocket?). This notebook flips that axis: it
is organised **per backend** (AlphaFold3, Boltz-2, Chai-1, OpenFold3,
Protenix, RosettaFold3) -- for a given model, how confident is it (pTM) and
how much of conformational space does it actually sample, across every
protein `results/tm_alignment/` currently has data for?

Because each per-backend cell below loops over every protein discovered at
run time (see the "Protein discovery" cell), **no new cells are needed as
more proteins land** -- unlike `tm_conformation_clustering.ipynb`'s
per-protein cells, which are added one pair at a time as
`worflows/postprocessing/Snakefile` (stage 6, `scripts/tm_helix_alignment.py`)
finishes each protein x form. Just re-run the notebook top to bottom.

Two signals per backend, both computed on a **shared per-protein PCA
embedding** (fit once on that protein's full pooled ensemble across all 6
backends, see the "Precompute embeddings" cell) so a backend's spread is
measured on the same axes every other backend is measured on, for that
protein:

- **Confidence** -- pTM, used to colour the PCA scatter below (falls back to
  `rmsd_tm`, per-frame RMSD to the ensemble mean, for a backend/protein
  combination where no pTM was found -- see `_confidence_col_for_backend`
  below), plus mean/std per protein in the summary table.
- **Conformational diversity** -- how far that backend's own points spread
  in the shared 2-D PCA space (a scalar "spread", the sqrt of the trace of
  their covariance) and how many distinct HDBSCAN clusters they fall into
  (noise excluded) -- a model that always collapses to the same structure
  will have low spread and 1 cluster regardless of how many seeds it ran.

A final cross-backend comparison cell turns both signals into one
leaderboard across all 6 models.


In [1]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA

ROOT       = Path("..")
ALIGN_ROOT = ROOT / "results" / "tm_alignment"

# Same 6-backend palette as tm_conformation_clustering.ipynb, so a given
# model is drawn in the same colour in both notebooks.
MODEL_PALETTE = {
    "alphafold3":   "#1f77b4",
    "boltz":        "#ff7f00",
    "chai1":        "#2ca02c",
    "openfold3":    "#d62728",
    "protenix":     "#9467bd",
    "rosettafold3": "#8c564b",
}

# Mirrors scripts/tm_helix_alignment.py's BACKEND_PATTERNS -- duplicated
# here (not imported) so this notebook stays self-contained, same
# convention tm_conformation_clustering.ipynb uses. Keep in sync if the
# script's version changes.
BACKEND_PATTERNS = list(MODEL_PALETTE)


def _load_run(run_name):
    """Load one apo/holo run's aligned TM-Ca ensemble + per-frame metadata
    (model/backend, seed, sample index, pTM), written by
    scripts/tm_helix_alignment.py -- already pooled across all 6 ABCfold
    backends x seed x diffusion/sample for this run."""
    npy = ALIGN_ROOT / run_name / "aligned_ca_tm.npy"
    csv = ALIGN_ROOT / run_name / "meta.csv"
    if not npy.exists():
        raise FileNotFoundError(
            f"{npy} not found -- run worflows/postprocessing/Snakefile "
            f"(scripts/tm_helix_alignment.py) for {run_name} first")
    coords = np.load(npy)                          # (n_frames, n_ca_tm, 3)
    meta   = pd.read_csv(csv)
    X      = coords.reshape(coords.shape[0], -1)   # flatten to (n_frames, n_ca_tm*3)
    return X, meta


def _kabsch_fit(P, Q):
    """Rotation R (3,3) and translation t (3,) such that (R @ P.T).T + t ~= Q.
    Same as kabsch() in scripts/tm_helix_alignment.py."""
    p_mean, q_mean = P.mean(axis=0), Q.mean(axis=0)
    Pc, Qc = P - p_mean, Q - q_mean
    U, _, Vt = np.linalg.svd(Pc.T @ Qc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    t = q_mean - R @ p_mean
    return R, t


def _load_protein(protein):
    """Load the aligned, multi-backend TM-Ca ensemble for one BASE protein
    (e.g. 'NPF2.12_Q9LFX9'), merging its apoform and holoform ABCfold runs
    into a single pooled ensemble. Same logic as
    tm_conformation_clustering.ipynb's _load_protein: holo's mean TM
    structure is Kabsch-refit onto apo's mean TM structure (apo is the
    anchor since it always exists) before pooling, since apo and holo are
    independent ABCfold jobs with no shared global orientation."""
    X_parts, meta_parts = [], []
    apo_mean = None
    for status in ("apo", "holo"):
        run_name = f"{protein}__{status}"
        if not (ALIGN_ROOT / run_name).exists():
            if status == "apo":
                raise FileNotFoundError(
                    f"{ALIGN_ROOT / run_name} not found -- apoform is expected "
                    f"for every protein; run worflows/postprocessing/Snakefile first")
            continue
        X, meta = _load_run(run_name)
        coords = X.reshape(X.shape[0], -1, 3)  # (n_frames, n_ca_tm, 3)

        if status == "apo":
            apo_mean = coords.mean(axis=0)
        else:
            R, t = _kabsch_fit(coords.mean(axis=0), apo_mean)
            flat = coords.reshape(-1, 3)
            coords = ((R @ flat.T).T + t).reshape(coords.shape)
            X = coords.reshape(coords.shape[0], -1)

        meta = meta.copy()
        meta["status"] = status
        X_parts.append(X)
        meta_parts.append(meta)

    X    = np.concatenate(X_parts, axis=0)
    meta = pd.concat(meta_parts, ignore_index=True)
    return X, meta


print("Setup done.")


Setup done.


## Protein discovery

Every protein under `results/tm_alignment/` at run time, apo/holo suffix
stripped back to the BASE name `_load_protein` expects.

In [2]:
def _base_protein_name(dirname):
    if dirname.endswith("__apo") or dirname.endswith("__holo"):
        return dirname.rsplit("__", 1)[0]
    return dirname


PROTEINS = sorted({
    _base_protein_name(p.name) for p in ALIGN_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith(".")
})

print(f"{len(PROTEINS)} protein(s) under results/tm_alignment/: {PROTEINS}")


1 protein(s) under results/tm_alignment/: ['NPF2.12_Q9LFX9']


## Precompute embeddings

Fit one 2-D PCA per protein on its **full** pooled ensemble (all 6 backends
together, same as `tm_conformation_clustering.ipynb`'s `plot_pca`), once,
here -- every per-backend cell below reuses these `pc1`/`pc2` coordinates
rather than re-fitting PCA per backend, so a backend's spread is always
measured on the same axes as every other backend for that protein. Re-run
this cell (and everything below it) after new proteins land.

In [3]:
PROTEIN_DATA = {}  # protein -> {"meta": DataFrame w/ pc1,pc2 cols, "evr": explained_variance_ratio, "n_atoms": int}

for protein in PROTEINS:
    X, meta = _load_protein(protein)
    n_pc = min(2, X.shape[1])
    pca = PCA(n_components=n_pc)
    coords = pca.fit_transform(X)
    meta = meta.copy()
    meta["pc1"] = coords[:, 0]
    meta["pc2"] = coords[:, 1] if n_pc > 1 else 0.0
    PROTEIN_DATA[protein] = {
        "meta": meta,
        "evr": pca.explained_variance_ratio_,
        "n_atoms": X.shape[1] // 3,
    }
    print(f"  {protein:20s} {len(meta):4d} frames  "
          f"PC1+PC2 explain {pca.explained_variance_ratio_[:2].sum():.1%} of variance")


  NPF2.12_Q9LFX9       4621 frames  PC1+PC2 explain 91.5% of variance


## Per-backend capability metrics

`_backend_diversity` -- for one backend's frames within one protein's shared
PCA embedding: `spread` (sqrt of the trace of their covariance in PC1/PC2 --
a bigger number means that backend's structures are more spread out, i.e.
more conformationally diverse) and `n_clusters` (a quick HDBSCAN fit,
`min_cluster_size` scaled to 10% of that backend's frame count for this
protein, noise excluded -- needs at least `min_frames` frames to attempt,
else `NaN`).

`_backend_summary_table` -- one row per protein for a given backend: frame
count, pTM mean/std, RMSD-to-mean mean/std, spread, n_clusters. `NaN`/0 rows
mean that backend has no frames for that protein (yet).

`_confidence_col_for_backend` -- pTM is the preferred confidence signal, but
`find_confidence()` in `scripts/tm_helix_alignment.py` doesn't always find a
parseable confidence file for every backend (at the time of writing, none of
this pipeline's 6 backends have any -- every `ptm` value is `NaN`). Rather
than plot an empty histogram/uncoloured scatter when that happens, this
falls back to `rmsd_tm` (per-frame RMSD in Å to that ensemble's converged
TM-helix mean -- always populated by `tm_helix_alignment.py`) as the
confidence/quality proxy for that backend, and says so.

`plot_backend_capability` -- the cell every per-backend section below calls:
the summary table, then a small-multiples PCA scatter (one panel per protein
that backend has data for, points coloured by pTM -- or RMSD-to-mean when
pTM is unavailable for this backend) showing where in that protein's shared
embedding this backend's structures actually land.

In [4]:
# color_by name -> (axis/legend label, colorscale, (cmin, cmax) or None for data-range)
CONFIDENCE_CONFIG = {
    "ptm":     ("pTM",                          "Viridis", (0.0, 1.0)),
    "rmsd_tm": ("RMSD to ensemble mean (Å)", "Plasma",  None),
}


def _confidence_col_for_backend(backend):
    """'ptm' if any protein has a non-NaN pTM value for this backend, else
    'rmsd_tm' (always populated by scripts/tm_helix_alignment.py)."""
    for protein in PROTEINS:
        sub = PROTEIN_DATA[protein]["meta"]
        sub = sub[sub["model"] == backend]
        if sub["ptm"].notna().any():
            return "ptm"
    return "rmsd_tm"


def _backend_diversity(sub_meta, min_frames=15, min_cluster_frac=0.1, min_samples=5):
    n = len(sub_meta)
    if n < 2:
        return np.nan, np.nan, n
    xy = sub_meta[["pc1", "pc2"]].to_numpy()
    spread = float(np.sqrt(np.trace(np.cov(xy, rowvar=False))))
    if n < min_frames:
        return spread, np.nan, n
    min_cluster_size = max(3, int(round(n * min_cluster_frac)))
    labels = HDBSCAN(min_cluster_size=min_cluster_size,
                      min_samples=min(min_samples, n - 1), copy=False).fit(xy).labels_
    n_clusters = len(set(l for l in labels if l >= 0))
    return spread, n_clusters, n


def _backend_summary_table(backend):
    rows = []
    for protein in PROTEINS:
        meta = PROTEIN_DATA[protein]["meta"]
        sub = meta[meta["model"] == backend]
        if sub.empty:
            rows.append({"protein": protein, "n_frames": 0, "ptm_mean": np.nan, "ptm_std": np.nan,
                         "rmsd_tm_mean": np.nan, "rmsd_tm_std": np.nan,
                         "spread": np.nan, "n_clusters": np.nan})
            continue
        spread, n_clusters, n = _backend_diversity(sub)
        rows.append({
            "protein": protein, "n_frames": n,
            "ptm_mean": float(sub["ptm"].mean()), "ptm_std": float(sub["ptm"].std()),
            "rmsd_tm_mean": float(sub["rmsd_tm"].mean()), "rmsd_tm_std": float(sub["rmsd_tm"].std()),
            "spread": spread, "n_clusters": n_clusters,
        })
    return pd.DataFrame(rows)


def plot_backend_capability(backend, marker_size=6, opacity=0.7):
    """Confidence + conformational-diversity snapshot for one ABCfold
    backend, pooled across every protein under results/tm_alignment/.

    backend  one of BACKEND_PATTERNS ('alphafold3', 'boltz', 'chai1',
             'openfold3', 'protenix', 'rosettafold3').
    """
    summary = _backend_summary_table(backend)
    display(summary)

    present = summary[summary["n_frames"] > 0]
    if present.empty:
        print(f"No {backend} frames yet for any protein.")
        return

    color_col = _confidence_col_for_backend(backend)
    label, colorscale, bounds = CONFIDENCE_CONFIG[color_col]
    if color_col != "ptm":
        print(f"[{backend}] no pTM found for any protein yet -- using {label} instead.")

    # -- small-multiples PCA scatter: one panel per protein, coloured by confidence --
    proteins_with_data = [p for p in PROTEINS if not PROTEIN_DATA[p]["meta"][PROTEIN_DATA[p]["meta"]["model"] == backend].empty]
    ncols = min(3, len(proteins_with_data))
    nrows = math.ceil(len(proteins_with_data) / ncols)
    if bounds is not None:
        cmin, cmax = bounds
    else:
        all_vals = pd.concat([
            PROTEIN_DATA[p]["meta"].loc[PROTEIN_DATA[p]["meta"]["model"] == backend, color_col]
            for p in proteins_with_data
        ])
        cmin, cmax = float(all_vals.min()), float(all_vals.max())
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=proteins_with_data)
    for i, protein in enumerate(proteins_with_data):
        row, col = i // ncols + 1, i % ncols + 1
        meta = PROTEIN_DATA[protein]["meta"]
        sub = meta[meta["model"] == backend]
        fig.add_trace(go.Scatter(
            x=sub["pc1"], y=sub["pc2"], mode="markers",
            marker=dict(size=marker_size, color=sub[color_col], colorscale=colorscale,
                        cmin=cmin, cmax=cmax, opacity=opacity,
                        showscale=(i == 0), colorbar=dict(title=label) if i == 0 else None),
            showlegend=False,
            text=[f"seed {s} sample {si}" for s, si in zip(sub["seed"], sub["sample_index"])],
            hovertemplate="%{text}<br>" + label + " %{marker.color:.3f}<extra></extra>",
        ), row=row, col=col)
    fig.update_layout(
        title=f"{backend}<br>PCA embedding per protein (shared per-protein PCA space, coloured by {label})",
        template="plotly_white", height=480 * nrows, width=520 * ncols, showlegend=False,
    )
    fig.show()


## Per-backend cells

One markdown + code cell pair per ABCfold backend. Each code cell is a
single `plot_backend_capability(...)` call -- edit its arguments directly
for one-off exploration on that backend without touching any other cell.

### `alphafold3`

Google DeepMind's AlphaFold3 -- the diffusion-based architecture the other 5 backends here are either reimplementations of or successors to; treated as the baseline.

In [5]:
plot_backend_capability("alphafold3")

,protein,n_frames,ptm_mean,ptm_std,rmsd_tm_mean,rmsd_tm_std,spread,n_clusters
0,NPF2.12_Q9LFX9,101,NaN,NaN,0.523031,0.426079,8.790644,2


[alphafold3] no pTM found for any protein yet -- using RMSD to ensemble mean (Å) instead.


### `boltz`

Boltz-2 (MIT) -- open, AF3-style diffusion model with an additional binding-affinity head.

In [6]:
plot_backend_capability("boltz")

,protein,n_frames,ptm_mean,ptm_std,rmsd_tm_mean,rmsd_tm_std,spread,n_clusters
0,NPF2.12_Q9LFX9,100,NaN,NaN,0.88669,0.319244,8.818318,2


[boltz] no pTM found for any protein yet -- using RMSD to ensemble mean (Å) instead.


### `chai1`

Chai-1 (Chai Discovery) -- open, AF3-style diffusion model trained independently of AlphaFold3.

In [7]:
plot_backend_capability("chai1")

,protein,n_frames,ptm_mean,ptm_std,rmsd_tm_mean,rmsd_tm_std,spread,n_clusters
0,NPF2.12_Q9LFX9,100,NaN,NaN,0.831335,0.225031,8.899929,3


[chai1] no pTM found for any protein yet -- using RMSD to ensemble mean (Å) instead.


### `openfold3`

OpenFold3 -- fully open reimplementation and retraining of the AF3 architecture.

In [8]:
plot_backend_capability("openfold3")

,protein,n_frames,ptm_mean,ptm_std,rmsd_tm_mean,rmsd_tm_std,spread,n_clusters
0,NPF2.12_Q9LFX9,4000,NaN,NaN,0.288959,0.436102,7.730204,2


[openfold3] no pTM found for any protein yet -- using RMSD to ensemble mean (Å) instead.


### `protenix`

Protenix (ByteDance) -- open reimplementation of the AF3 architecture.

In [9]:
plot_backend_capability("protenix")

,protein,n_frames,ptm_mean,ptm_std,rmsd_tm_mean,rmsd_tm_std,spread,n_clusters
0,NPF2.12_Q9LFX9,100,NaN,NaN,2.070174,1.117727,19.717249,2


[protenix] no pTM found for any protein yet -- using RMSD to ensemble mean (Å) instead.


### `rosettafold3`

RoseTTAFold3 (Baker lab) -- independent architecture lineage (RoseTTAFold), not an AF3 reimplementation.

In [10]:
plot_backend_capability("rosettafold3")

,protein,n_frames,ptm_mean,ptm_std,rmsd_tm_mean,rmsd_tm_std,spread,n_clusters
0,NPF2.12_Q9LFX9,220,NaN,NaN,0.34174,0.072252,1.514062,3


[rosettafold3] no pTM found for any protein yet -- using RMSD to ensemble mean (Å) instead.


## Cross-backend comparison

Turns every backend's `_backend_summary_table` into one leaderboard: frame
count and protein coverage, frame-weighted mean confidence (pTM where
available, `rmsd_tm` fallback otherwise -- see `confidence_metric` column
and note pTM and RMSD point in opposite directions: higher pTM is better,
lower RMSD is better), mean conformational spread, and mean cluster count
(across proteins with enough frames to attempt HDBSCAN) -- one bar per
backend, using the same colours as `tm_conformation_clustering.ipynb`'s
`MODEL_PALETTE`.

In [11]:
def plot_backend_comparison():
    rows = []
    for backend in BACKEND_PATTERNS:
        summary = _backend_summary_table(backend)
        present = summary[summary["n_frames"] > 0]
        if present.empty:
            rows.append({"model": backend, "n_proteins": 0, "n_frames_total": 0,
                         "confidence_metric": None, "confidence_mean": np.nan,
                         "spread_mean": np.nan, "n_clusters_mean": np.nan})
            continue
        color_col = _confidence_col_for_backend(backend)
        rows.append({
            "model": backend,
            "n_proteins": int(len(present)),
            "n_frames_total": int(present["n_frames"].sum()),
            "confidence_metric": color_col,
            "confidence_mean": float(np.average(present[f"{color_col}_mean"], weights=present["n_frames"])),
            "spread_mean": float(present["spread"].mean()),
            "n_clusters_mean": (float(present["n_clusters"].dropna().mean())
                                if present["n_clusters"].notna().any() else np.nan),
        })
    comparison = pd.DataFrame(rows)
    display(comparison)

    colors = [MODEL_PALETTE[b] for b in comparison["model"]]
    fig = make_subplots(rows=1, cols=3, subplot_titles=("mean confidence (pTM / rmsd_tm)", "mean spread", "mean n clusters"))
    fig.add_trace(go.Bar(x=comparison["model"], y=comparison["confidence_mean"], marker_color=colors, showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=comparison["model"], y=comparison["spread_mean"], marker_color=colors, showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=comparison["model"], y=comparison["n_clusters_mean"], marker_color=colors, showlegend=False), row=1, col=3)
    fig.update_layout(
        title="Backend comparison, pooled across all proteins",
        template="plotly_white", height=520, width=1400,
    )
    fig.show()
    return comparison


_ = plot_backend_comparison()


,model,n_proteins,n_frames_total,confidence_metric,confidence_mean,spread_mean,n_clusters_mean
0,alphafold3,1,101,rmsd_tm,0.523031,8.790644,2.0
1,boltz,1,100,rmsd_tm,0.886690,8.818318,2.0
2,chai1,1,100,rmsd_tm,0.831335,8.899929,3.0
3,openfold3,1,4000,rmsd_tm,0.288959,7.730204,2.0
4,protenix,1,100,rmsd_tm,2.070174,19.717249,2.0
5,rosettafold3,1,220,rmsd_tm,0.341740,1.514062,3.0
